# Import

### libraries

In [ ]:
import sys
import os
import warnings
import random
import datetime
import itertools

import pandas as pd
import numpy as np
# viz
import matplotlib.pyplot as plt
# statistics
from scipy.stats import boxcox
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

In [ ]:
sys.path.append("../../")


from ML.metric.regression import root_mean_squared_log_error

### env & settings

In [ ]:
DATA_DIR = "/workspace/Storage/kaggle/Data/insurance"
PATH_TRAIN = os.path.join(DATA_DIR, "raw", "train.csv")
PATH_TEST = os.path.join(DATA_DIR, "raw", "test.csv")

In [ ]:
# 경고 출력 끄기
warnings.filterwarnings('ignore')

# 출력할 컬럼 수를 충분히 늘리기
pd.set_option('display.max_columns', None)

In [ ]:
seed = 42
random.seed(seed)


### Data

In [ ]:
train = pd.read_csv(PATH_TRAIN)
test = pd.read_csv(PATH_TEST)
EDA_result = pd.read_csv("EDA_result.csv")

In [ ]:
data_type = EDA_result[EDA_result['transform'] != 'drop'][['name','type']].set_index('name').to_dict()['type']
intervals = EDA_result[EDA_result['type'] == 'interval']['name'].to_list()[1:]
ratios = EDA_result[EDA_result['type'] == 'ratio']['name'].to_list()
ordinals = EDA_result[EDA_result['type'] == 'ordinal']['name'].to_list()
nominals = EDA_result[EDA_result['type'] == 'nominal']['name'].to_list()
categorical = nominals + ordinals
numeric = ordinals + intervals + ratios
target = "Premium Amount"
len(intervals), len(ratios), len(ordinals), len(nominals), 

(2, 8, 4, 6)

In [ ]:
# 변경
def preprocess_id(df):
    df = df.drop('id', axis=1)
    return df

def preprocess_Policy_Start_Date_1(df):
    df['Policy Start Date'] = pd.to_datetime(df['Policy Start Date'])
    return df

def preprocess_Policy_Start_Date_2(df):
    ref = "2024-08-16"
    ref = datetime.datetime.strptime(ref, "%Y-%m-%d")
    df["Policy Start Date"] = df["Policy Start Date"].map(lambda x: (ref-x).total_seconds())
    return df

def preprocess_Policy_Start_Date_3(df):
    df = df.select_dtypes(include=['number','datetime'])
    df = df.select_dtypes(include=['number','datetime'])
    df = df.set_index("Policy Start Date")
    df = df.resample('D').mean()
    return df

def preprocess_Premium_Amount_1(df):
    # box-cox?
    df["Premium Amount"] = boxcox(df["Premium Amount"])[0]
    return df

def scaling_standardization(s):
    mean = s.mean()
    std = s.std()
    return s.map(lambda x: (x-mean)/std)

def scaling_minmax(s):
    min_ = s.min()
    max_ = s.max()
    return s.map(lambda x: (x-min_)/max_)

def preprocess_scaling(df):
    booleans = list(filter(lambda x: len(x)==2, df.columns))
    remains = list(filter(lambda x: len(x)!=2, df.columns))
    df[booleans] = df[booleans].apply(scaling_minmax)
    df[remains] = df[remains].apply(scaling_standardization)
    return df

# 결측처리
def process_mv_baseline(df, EDA_result):
    EDA_result = EDA_result.set_index('name').to_dict()
    for col in df.columns:
        if EDA_result["transform"][col] == "drop":
            df.drop(col, axis=1, inplace=True)
        # elif EDA_result["mv_process"][col] == "special":
        #     df.drop(col, axis=1, inplace=True)
    df = df.dropna()
    return df

def process_mv_analysis(df, EDA_result):
    EDA_result = EDA_result.set_index('name').to_dict()
    for col in df.columns:
        if EDA_result["transform"][col] == "drop":
            df.drop(col, axis=1, inplace=True)
    return df

def process_mv_advanced(df):
    return df.dropna()

# 인코딩
def preprocess_ordinals_1(df,d):
    cols = list(d.keys())
    df[cols] = df[cols].apply(lambda col: col.map(lambda x: d[col.name][x] if not pd.isna(x) else x))
    return df

def preprocess_nominals_1(df, EDA_result):
    nominals = EDA_result[EDA_result['type']=="nominal"]['name'].to_list()
    nominals = [x for x in nominals if x in df.columns]
    df = pd.get_dummies(df, columns = nominals)
    return df


def preprocess_nominals_2(df, EDA_result):
    categoricals = EDA_result[EDA_result['type'].isin(["nominal", "ordinal"])]['name'].to_list()
    categoricals = [x for x in categoricals if x in df.columns]
    return pd.concat([df, pd.get_dummies(df[categoricals].astype(str))], axis=1)

ordinal_dict = {
    "Education Level" : {
        "High School": 1,
        "Bachelor's" : 2,
        "Master's" : 3,
        "PhD" : 4,
    },
    "Policy Type": {
        "Basic":1,
        "Comprehensive":2,
        "Premium":3
    },
    "Customer Feedback":{
        "Poor":1,
        "Average":2,
        "Good":3
    },
    "Exercise Frequency":{
        'Rarely':1,
        'Monthly':2,
        'Weekly':3,
        'Daily':4
    }
}

def pipeline_0(df, d, EDA_result):
    df_ = df.copy()
    p1 = preprocess_id(df_)
    p2 = preprocess_Policy_Start_Date_1(p1)
    p3 = preprocess_Policy_Start_Date_2(p2)
    p4 = preprocess_ordinals_1(p3,d)
    p5 = process_mv_baseline(p4, EDA_result)
    p6 = preprocess_nominals_1(p5, EDA_result)
    return p6

def pipeline_1(df, d):
    df_ = df.copy()
    p1 = preprocess_id(df_)
    p2 = preprocess_Policy_Start_Date_1(p1)
    p3 = preprocess_Policy_Start_Date_2(p2)
    p4 = preprocess_ordinals_1(p3,d)
    return p4

def pipeline_2(df, d, EDA_result):
    df_ = df.copy()
    p1 = preprocess_id(df_)
    p2 = preprocess_Policy_Start_Date_1(p1)
    p3 = preprocess_Policy_Start_Date_2(p2)
    p4 = preprocess_ordinals_1(p3,d)
    p5 = preprocess_nominals_1(p4, EDA_result)
    return p5



def pipeline_3(df, d, EDA_result):
    df_ = df.copy()
    p1 = preprocess_id(df_)
    p2 = preprocess_Policy_Start_Date_1(p1)
    p3 = preprocess_Policy_Start_Date_2(p2)
    p4 = preprocess_ordinals_1(p3,d)
    p5 = preprocess_nominals_2(p4, EDA_result)
    return p5

def pipeline_4(df, d, EDA_result):
    df_ = df.copy()
    p1 = preprocess_id(df_)
    p2 = preprocess_Policy_Start_Date_1(p1)
    p3 = preprocess_Policy_Start_Date_2(p2)
    p4 = preprocess_ordinals_1(p3,d)
    p5 = preprocess_nominals_1(p4, EDA_result)
    p6 = process_mv_advanced(p5)
    p7 = preprocess_scaling(p6)
    return p7

def pipeline_5(df, d):
    df_ = df.copy()
    p1 = preprocess_id(df_)
    p2 = preprocess_Policy_Start_Date_1(p1)
    p3 = preprocess_ordinals_1(p2,d)
    p4 = preprocess_Policy_Start_Date_3(p3)
    return p4

In [ ]:
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import LabelEncoder

def process_impute_mice(df, max_iter=10, categoricals = []):
    # mice는 categorical 변수를 읽지 못함. 하지만 ordinal 인코딩 하더라도 mice 알고리즘은 순서없이 계산하므로 ordinal 인코딩 후 계산하고 역변환
    if categoricals:
        # 1. Nominal 변수 Label Encoding
        les = {}
        for cat in categoricals:
            le = LabelEncoder()
            df[cat] = le.fit_transform(df[cat].fillna("nan"))  # NaN 포함 처리
            classes = dict(((str(y),x) for x,y in enumerate(le.classes_)))
            if classes.get("nan",""):
                nan_encoded = classes["nan"]
                df[cat] = df[cat].replace(nan_encoded, np.nan)
            les[cat] = le

    # MICE를 사용한 결측값 대체
    imputer = IterativeImputer(max_iter=max_iter, random_state=0)
    imputed_data = imputer.fit_transform(df)
    imputed_df = pd.DataFrame(imputed_data, columns=df.columns)

    # 역변환
    if categoricals:
        for col in categoricals:
            imputed_df[col] = imputed_df[col].round().astype(int)  # 범주형 값 정수화
            imputed_df[col] = les[col].inverse_transform(imputed_df[col])

    return imputed_df

In [ ]:
def pipeline_7(df, d, EDA_result, MARS):
    df_ = df.copy()
    p1 = preprocess_id(df_)
    p2 = preprocess_Policy_Start_Date_1(p1)
    p3 = preprocess_Policy_Start_Date_2(p2)
    p4 = preprocess_ordinals_1(p3,d)
    p5 = process_impute_mice(p4, 10, EDA_result[EDA_result["type"]=='nominal']['name'].tolist())
    p4[MARS] = p5[MARS]
    p4 = p4.dropna()
    p6 = preprocess_nominals_1(p4, EDA_result)
    p7 = preprocess_scaling(p6)
    return p7

In [ ]:
# 차원축소, 군집은 encoding과 missing value 처리가 필요. 군집에 최대한 정보를 반영하기 위해 MAR과 결측비율이 큰 것은 mice로 대체, 나머지 드랍. ordinal/원핫인코딩
MARS = EDA_result[EDA_result["mv_process"] == "special"]['name'].tolist() + ["Annual Income"]
train_p8 = pipeline_7(train, ordinal_dict, EDA_result, MARS)

# 1. exp